In [1]:
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json


from pathlib import Path

# Preliminari

In [2]:
# Configurazione OpenAI
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.TUNED_GPT_4_1_OVERSAMPLING
RESULTS_FILE_NAME = 'results_' + 'gpt-4.1-tuned-oversampling' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 65
len(validation_data) = 66


In [5]:
print(validation_data.iloc[0].report_text)

A LIVELLO DEL RETTO MEDIO-ALTO, A CIRCA 10 CM DAL MARGINE ANALE ESTERNO, SI DOCUMENTA GROSSOLANO ISPESSIMENTO PARIETALE CHE INTERESSA CIRCONFERENZIALMENTE IL VISCERE IN SEDE CENTRALE (CON ASPETTO STENOTICO) ED I 3/4 DELLA CIRCONFERENZA IN SEDE MARGINALE SUPERIORE ED INFERIORE, CON SPESSORE MASSIMO DI 27 MM ED ESTENSIONE LONGITUDINALE DI CIRCA 65 MM. L'ISPESSIMENTO E CARATTERIZZATO DA SEGNALE IPOINTENSO NELLE IMMAGINI T2-DIPENDENTI E MARCATA RESTRIZIONE DELLA DIFFUSIONE, CON SIGNIFICATO PATOLOGICO, E MOSTRA ESTESI SEGNI DI ESTENSIONE EXTRA-PARIETALE NEL SUO III MEDIO LUNGO IL VERSANTE ANTERIO-LATERALE SINISTRO (CON GROSSOLANO GETTONE SOLIDO DI 24 MM) OVE IL TESSUTO PATOLOGICO RAGGIUNGE ED INFILTRA LA FASCIA MESORETTALE E LA RIFLESSIONE PERITONEALE. A TALE LIVELLO IL TESSUTO GIUNGE IN ADIACENZA AL PROFILO POSTERIORE DEL CORPO UTERINO, CON APPARENTE CONSERVATO PIANO DI CLIVAGGIO ADIPOSO. 
MULTIPLI LINFONODI (> 6) A MORFOLOGIA ROTONDEGGIANTE ED ASSE CORTO MASSIMO DI 6 MM IN SEDE MESORETTAL

# Generazione

## Preliminari generazione

In [6]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=output_format
    )
    return response

In [7]:
# Esempio
if False:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [8]:
if False: # To see the full output
    pprint(result.model_dump())
if False:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

## Baseline inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [9]:
print(MODEL)
df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    actual.append(real.model_dump(mode='json'))
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.output_parsed.model_dump(mode='json'))

ft:gpt-4.1-2025-04-14:luca-tramonti:report-extractor:D8nTVX9B


  0%|          | 0/131 [00:00<?, ?it/s]

100%|██████████| 131/131 [11:28<00:00,  5.26s/it]


In [10]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [11]:
output.model_dump()

{'id': 'resp_027ebb5b9119f85900698f4888aadc81a293b8a86f0eb1f73b',
 'created_at': 1770997896.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'ft:gpt-4.1-2025-04-14:luca-tramonti:report-extractor:D8nTVX9B',
 'object': 'response',
 'output': [{'id': 'msg_027ebb5b9119f85900698f48893d2481a2997b9b1fc937bf08',
   'content': [{'annotations': [],
     'text': '{"morfologia":"solido_anulare","ore_inizio":null,"ore_fine":null,"spessore_parietale":null,"estensione_cranio_caudale":40,"distanza_oai":50,"posizione":{"basso":"no","medio":"si","alto":"si","giunzione":"no"},"riflessione_peritoneale_anteriore":"cavallo","infiltrazione_tessuto_adiposo":"si_5mm_plus","infiltrazione_sfinteri":"no","infiltrazione_organi_extra":"no","infiltrazione_organi_dettagli":{"pavimento_pelvico":"no","altro":"no"},"coinvolgimento_riflessione_peritoneale":"si","coinvolgimento_fascia_mesorettale":"si","numero_linfonodi_non_conosciuto":"conosciuto","linfonodi_sospetti":2,"s